# Part 3: Theoretical Calculators

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from calculators.base import GPUS_SPEC, ModelConfig, TrainingConfig
from calculators.baseline_calculator import BaselineCalculator
from calculators.efficient_calculator import EfficientCalculator

In [2]:
mc = ModelConfig(
    vocab_size=16000,
    hidden_dim=512,
    num_heads=8,
    num_layers=6,
    intermediate_dim=1024,
    max_seq_len=4096,
)
tc = TrainingConfig(batch_size=2, seq_len=4096, num_gpus=2)

gpu = GPUS_SPEC
print(f"GPU: {gpu.name}")
print(f"  Memory BW: {gpu.memory_bandwidth_gbps} GB/s")
print(f"  BF16 TFLOPS: {gpu.flops_bf16}")
print(f"  Interconnect BW: {gpu.interconnect_bandwidth_gbps} GB/s")

bc = BaselineCalculator(mc, tc, gpu)
ec = EfficientCalculator(mc, tc, gpu)

GPU: A800 SXM 80GB
  Memory BW: 2000 GB/s
  BF16 TFLOPS: 312
  Interconnect BW: 400 GB/s


## 3.1 Memory Calculator

In [3]:
def mb(x):
    return x / 1024**2

print(f"Total params: {bc.calculate_total_params():,}")
print()
print("=== Per-GPU Memory Breakdown (MB) ===")
print(f"{'Component':<25} {'Baseline':>12} {'Efficient':>12}")
print("-" * 51)
print(f"{'Parameters':<25} {mb(bc.calculate_param_memory()):>12.1f} {mb(ec.calculate_param_memory()):>12.1f}")
print(f"{'Gradients':<25} {mb(bc.calculate_gradient_memory()):>12.1f} {mb(ec.calculate_gradient_memory()):>12.1f}")
print(f"{'Optimizer states':<25} {mb(bc.calculate_optimizer_memory()):>12.1f} {mb(ec.calculate_optimizer_memory()):>12.1f}")
fsdp_buf = mb(ec.calculate_fsdp_buffer_memory())
print(f"{'FSDP buffers':<25} {'N/A':>12} {fsdp_buf:>12.1f}")
print(f"{'Activations':<25} {mb(bc.calculate_activation_memory()):>12.1f} {mb(ec.calculate_activation_memory()):>12.1f}")
print("-" * 51)
print(f"{'Peak memory':<25} {mb(bc.calculate_peak_memory()):>12.1f} {mb(ec.calculate_peak_memory()):>12.1f}")
print()
print(f"Peak memory reduction: {bc.calculate_peak_memory() / ec.calculate_peak_memory():.2f}x")

Total params: 32,119,296

=== Per-GPU Memory Breakdown (MB) ===
Component                     Baseline    Efficient
---------------------------------------------------
Parameters                       183.8         61.3
Gradients                        122.5         61.3
Optimizer states                 367.6        183.8
FSDP buffers                       N/A         20.0
Activations                     9770.8        544.4
---------------------------------------------------
Peak memory                    10444.7        870.7

Peak memory reduction: 12.00x


### Activation memory per operation

In [5]:
B, S, H, I, V, L = tc.batch_size, tc.seq_len, mc.hidden_dim, mc.intermediate_dim, mc.vocab_size, mc.num_layers
n_heads = mc.num_heads
d = tc.dtype_bytes

baseline_acts = {
    "RMSNorm (x2)": 2 * (
        B * S * H * d
        + B * S * H * 4
        + B * S * H * 4
        + B * S * 1 * 4
        + B * S * 1 * 4
        + B * S * H * 4
    ),
    "Attention": (
        3 * B * S * H * d
        + 3 * B * S * H * d
        + 2 * B * S * H * 4
        + 2 * B * S * H * d
        + 2 * B * n_heads * S * S * d
        + 2 * B * S * H * d
    ),
    "SwiGLU MLP": (
        2 * B * S * H * d
        + 8 * B * S * I * d
    ),
    "Residuals": 2 * B * S * H * d,
    "Loss (logits)": B * S * V * d + 2 * B * S * V * 4,
}

efficient_acts = {
    "RMSNorm (x2)": 2 * (
        B * S * H * d
        + B * S * 1 * 4
    ),
    "Attention": 6 * B * S * H * d,
    "SwiGLU MLP": B * S * H * d,
    "Residuals": 2 * B * S * H * d,
    "Loss (fused)": 0,
}

print(f"{'Operation':<25} {'Baseline (MB)':>14} {'Efficient (MB)':>14} {'Ratio':>8}")
print("-" * 63)
for key in baseline_acts:
    bv = baseline_acts[key]
    ek = key.replace("(logits)", "(fused)") if "logits" in key else key
    ev = efficient_acts.get(ek, efficient_acts.get(key, 0))
    ratio = bv / ev if ev > 0 else float('inf')
    print(f"{key:<25} {mb(bv):>14.1f} {mb(ev):>14.1f} {ratio:>7.1f}x")

b_total = sum(baseline_acts.values()) * L + baseline_acts["Loss (logits)"]
e_total = sum(efficient_acts.values()) * L
print("-" * 63)
print(f"per-layer values shown. Total = per-layer * {L} layers + embedding + final norm + loss.")

Operation                  Baseline (MB) Efficient (MB)    Ratio
---------------------------------------------------------------
RMSNorm (x2)                       112.1           16.1     7.0x
Attention                         1136.0           48.0    23.7x
SwiGLU MLP                         144.0            8.0    18.0x
Residuals                           16.0           16.0     1.0x
Loss (logits)                     1250.0            0.0     infx
---------------------------------------------------------------
per-layer values shown. Total = per-layer * 6 layers + embedding + final norm + loss.


## 3.2 Time and Comms Calculator

In [6]:
print("=== Per-Operation Forward Time (ms) ===")
print(f"{'Operation':<25} {'Baseline':>12} {'Efficient':>12}")
print("-" * 51)
print(f"{'Embedding':<25} {bc.time_embedding_ms():>12.4f} {ec.time_embedding_ms():>12.4f}")
print(f"{'RMSNorm':<25} {bc.time_rms_norm_ms():>12.4f} {ec.time_rms_norm_ms():>12.4f}")
print(f"{'Attention':<25} {bc.time_attention_ms():>12.4f} {ec.time_attention_ms():>12.4f}")
print(f"{'MLP':<25} {bc.time_mlp_ms():>12.4f} {ec.time_mlp_ms():>12.4f}")
print(f"{'LM Head':<25} {bc.time_lm_head_ms():>12.4f} {ec.time_lm_head_ms():>12.4f}")
print(f"{'Loss':<25} {bc.time_loss_ms():>12.4f} {ec.time_loss_ms():>12.4f}")
print("-" * 51)
print(f"{'Forward pass':<25} {bc.time_forward_pass_ms():>12.4f} {ec.time_forward_pass_ms():>12.4f}")
print(f"{'Backward pass':<25} {bc.time_backward_pass_ms():>12.4f} {ec.time_backward_pass_ms():>12.4f}")
print(f"{'Forward + Backward':<25} {bc.time_forward_backward_ms():>12.4f} {ec.time_forward_backward_ms():>12.4f}")

=== Per-Operation Forward Time (ms) ===
Operation                     Baseline    Efficient
---------------------------------------------------
Embedding                       0.0124       0.0124
RMSNorm                         0.0084       0.0084
Attention                       0.3030       0.2753
MLP                             0.0829       0.0829
LM Head                         0.4302       0.0000
Loss                            0.2622       0.4323
---------------------------------------------------
Forward pass                    3.1292       2.7028
Backward pass                   6.2584       5.4057
Forward + Backward              9.3877       8.1085


In [7]:
print("=== Communication ===")
print(f"{'Metric':<35} {'Baseline (DDP)':>14} {'Efficient (FSDP)':>16}")
print("-" * 67)
print(f"{'Comm volume (MB)':<35} {mb(bc.calculate_communication_volume()):>14.1f} {mb(ec.calculate_communication_volume()):>16.1f}")
print(f"{'Comm time (ms)':<35} {bc.time_communication_ms():>14.4f} {ec.time_communication_ms():>16.4f}")
print(f"{'Overlap efficiency':<35} {bc.overlap_efficiency():>14.1%} {ec.overlap_efficiency():>16.1%}")
print(f"{'Non-overlapped comm (ms)':<35} {(1 - bc.overlap_efficiency()) * bc.time_communication_ms():>14.4f} {(1 - ec.overlap_efficiency()) * ec.time_communication_ms():>16.4f}")
print()
print("=== Total Step Time ===")
print(f"{'Baseline':<25} {bc.time_total_step_ms():>12.4f} ms")
print(f"{'Efficient':<25} {ec.time_total_step_ms():>12.4f} ms")
print(f"{'Speedup':<25} {bc.time_total_step_ms() / ec.time_total_step_ms():>12.2f}x")

print()
b_compute = bc.time_forward_backward_ms()
b_comm = bc.time_communication_ms()
e_compute = ec.time_forward_backward_ms()
e_comm = ec.time_communication_ms()
print(f"Baseline  compute/comm ratio: {b_compute / b_comm:.1f} ({'compute-bound' if b_compute > b_comm else 'comm-bound'})")
print(f"Efficient compute/comm ratio: {e_compute / e_comm:.1f} ({'compute-bound' if e_compute > e_comm else 'comm-bound'})")

=== Communication ===
Metric                              Baseline (DDP) Efficient (FSDP)
-------------------------------------------------------------------
Comm volume (MB)                             245.1            183.8
Comm time (ms)                              0.6424           0.4818
Overlap efficiency                           90.0%            90.0%
Non-overlapped comm (ms)                    0.0642           0.0482

=== Total Step Time ===
Baseline                        9.4519 ms
Efficient                       8.1567 ms
Speedup                           1.16x

Baseline  compute/comm ratio: 14.6 (compute-bound)
Efficient compute/comm ratio: 16.8 (compute-bound)


# Part 4: Report and Analysis

Run benchmark:

In [9]:
!export CUDA_VISIBLE_DEVICES=0,1 && torchrun --nproc_per_node=2 tests/bench_e2e.py

Running baseline training...
  Warmup run (compiling kernels)...
Running baseline training...
  Warmup run (compiling kernels)...
Training with 2 GPU(s)
Device: cuda:0
Model config: TransformerConfig(vocab_size=16000, hidden_dim=512, num_heads=8, num_layers=6, intermediate_dim=1024, max_seq_len=4096, dropout=0.0, rope_theta=10000.0, rms_norm_eps=1e-06)
Model parameters: 32,119,296

Training for 1 epochs (10 steps)
Batch size: 2 x 2 = 4
Sequence length: 4096
------------------------------------------------------------
Epoch 1 complete | Avg Loss: 9.7811
  Measurement run...
Training with 2 GPU(s)
Device: cuda:0
Model config: TransformerConfig(vocab_size=16000, hidden_dim=512, num_heads=8, num_layers=6, intermediate_dim=1024, max_seq_len=4096, dropout=0.0, rope_theta=10000.0, rms_norm_eps=1e-06)
Model parameters: 32,119,296

Training for 1 epochs (10 steps)
Batch size: 2 x 2 = 4
Sequence length: 4096
------------------------------------------------------------
Epoch 1 complete | Avg Loss

In [10]:
measured_baseline_ms_per_step = 95.55
measured_efficient_ms_per_step = 49.71
measured_baseline_peak_mb = 14415.5
measured_efficient_peak_mb = 1163.9

## Memory table

In [11]:
print("=== Memory Table: Per-Operation Activation Memory (per layer) ===")
print(f"{'Operation':<25} {'Baseline (MB)':>14} {'Efficient (MB)':>14} {'Savings':>10}")
print("-" * 65)

for key in baseline_acts:
    bv = baseline_acts[key]
    ek = key.replace("(logits)", "(fused)") if "logits" in key else key
    ev = efficient_acts.get(ek, efficient_acts.get(key, 0))
    ratio = bv / ev if ev > 0 else float('inf')
    print(f"{key:<25} {mb(bv):>14.1f} {mb(ev):>14.1f} {ratio:>9.1f}x")

print()
print("=== Peak Memory (calculated vs measured) ===")
print(f"{'':^25} {'Calculated':>12} {'Measured':>12} {'Diff':>10}")
print("-" * 61)
calc_b = mb(bc.calculate_peak_memory())
calc_e = mb(ec.calculate_peak_memory())
print(f"{'Baseline (MB)':<25} {calc_b:>12.1f} {measured_baseline_peak_mb:>12.1f} {(measured_baseline_peak_mb - calc_b) / calc_b:>9.1%}")
print(f"{'Efficient (MB)':<25} {calc_e:>12.1f} {measured_efficient_peak_mb:>12.1f} {(measured_efficient_peak_mb - calc_e) / calc_e:>9.1%}")
print()
print(f"Measured memory reduction: {measured_baseline_peak_mb / measured_efficient_peak_mb:.2f}x")
print(f"Calculated memory reduction: {bc.calculate_peak_memory() / ec.calculate_peak_memory():.2f}x")

=== Memory Table: Per-Operation Activation Memory (per layer) ===
Operation                  Baseline (MB) Efficient (MB)    Savings
-----------------------------------------------------------------
RMSNorm (x2)                       112.1           16.1       7.0x
Attention                         1136.0           48.0      23.7x
SwiGLU MLP                         144.0            8.0      18.0x
Residuals                           16.0           16.0       1.0x
Loss (logits)                     1250.0            0.0       infx

=== Peak Memory (calculated vs measured) ===
                            Calculated     Measured       Diff
-------------------------------------------------------------
Baseline (MB)                  10444.7      14415.5     38.0%
Efficient (MB)                   870.7       1163.9     33.7%

Measured memory reduction: 12.39x
Calculated memory reduction: 12.00x


## Time table

In [12]:
print("=== Time Table: Per-Operation Compute Time (ms, roofline) ===")
print(f"{'Operation':<25} {'Baseline':>12} {'Efficient':>12}")
print("-" * 51)
time_ops = [
    ("Embedding", bc.time_embedding_ms(), ec.time_embedding_ms()),
    ("RMSNorm", bc.time_rms_norm_ms(), ec.time_rms_norm_ms()),
    ("Attention", bc.time_attention_ms(), ec.time_attention_ms()),
    ("MLP (SwiGLU)", bc.time_mlp_ms(), ec.time_mlp_ms()),
    ("LM Head", bc.time_lm_head_ms(), ec.time_lm_head_ms()),
    ("Loss", bc.time_loss_ms(), ec.time_loss_ms()),
]
for name, bval, eval_ in time_ops:
    print(f"{name:<25} {bval:>12.4f} {eval_:>12.4f}")
print("-" * 51)
print(f"{'Total step (calculated)':<25} {bc.time_total_step_ms():>12.4f} {ec.time_total_step_ms():>12.4f}")

=== Time Table: Per-Operation Compute Time (ms, roofline) ===
Operation                     Baseline    Efficient
---------------------------------------------------
Embedding                       0.0124       0.0124
RMSNorm                         0.0084       0.0084
Attention                       0.3030       0.2753
MLP (SwiGLU)                    0.0829       0.0829
LM Head                         0.4302       0.0000
Loss                            0.2622       0.4323
---------------------------------------------------
Total step (calculated)         9.4519       8.1567


## Validation: calculated vs measured

In [13]:
print("=== Step Time: Calculated vs Measured ===")
print(f"{'':^25} {'Calculated':>12} {'Measured':>12} {'Ratio':>10}")
print("-" * 61)
calc_b = bc.time_total_step_ms()
calc_e = ec.time_total_step_ms()
print(f"{'Baseline (ms/step)':<25} {calc_b:>12.2f} {measured_baseline_ms_per_step:>12.2f} {measured_baseline_ms_per_step / calc_b:>9.1f}x")
print(f"{'Efficient (ms/step)':<25} {calc_e:>12.2f} {measured_efficient_ms_per_step:>12.2f} {measured_efficient_ms_per_step / calc_e:>9.1f}x")
print()
print(f"Measured speedup: {measured_baseline_ms_per_step / measured_efficient_ms_per_step:.2f}x")
print(f"Calculated speedup: {bc.time_total_step_ms() / ec.time_total_step_ms():.2f}x")

print()
print("=== Peak Memory: Calculated vs Measured ===")
print(f"{'':^25} {'Calculated':>12} {'Measured':>12} {'Ratio':>10}")
print("-" * 61)
calc_bm = mb(bc.calculate_peak_memory())
calc_em = mb(ec.calculate_peak_memory())
print(f"{'Baseline (MB)':<25} {calc_bm:>12.1f} {measured_baseline_peak_mb:>12.1f} {measured_baseline_peak_mb / calc_bm:>9.1f}x")
print(f"{'Efficient (MB)':<25} {calc_em:>12.1f} {measured_efficient_peak_mb:>12.1f} {measured_efficient_peak_mb / calc_em:>9.1f}x")
print()
print(f"Measured memory reduction: {measured_baseline_peak_mb / measured_efficient_peak_mb:.2f}x")
print(f"Calculated memory reduction: {bc.calculate_peak_memory() / ec.calculate_peak_memory():.2f}x")

=== Step Time: Calculated vs Measured ===
                            Calculated     Measured      Ratio
-------------------------------------------------------------
Baseline (ms/step)                9.45        95.55      10.1x
Efficient (ms/step)               8.16        49.71       6.1x

Measured speedup: 1.92x
Calculated speedup: 1.16x

=== Peak Memory: Calculated vs Measured ===
                            Calculated     Measured      Ratio
-------------------------------------------------------------
Baseline (MB)                  10444.7      14415.5       1.4x
Efficient (MB)                   870.7       1163.9       1.3x

Measured memory reduction: 12.39x
Calculated memory reduction: 12.00x


## Discrepancy analysis

### Memory

Calculated peak: baseline ~10.4 GB, efficient ~871 MB. Measured: ~14.4 GB и ~1.2 GB. Расхождение ~35-38%.

Основные причины:
- PyTorch autograd сохраняет больше промежуточных тензоров, чем мы учитываем (метаданные графа, версионные счётчики, маски из elementwise-операций)
- Фрагментация CUDA-аллокатора — блоки округляются до 2/20 МБ
- DDP/FSDP выделяют дополнительные буферы для коммуникации

При этом **ratio совпадает отлично**: 12.0x (calc) vs 12.4x (measured) — калькулятор верно моделирует относительный выигрыш.

### Time

Calculated step time: baseline ~9.5 ms, efficient ~8.2 ms. Measured: ~95.6 ms и ~49.7 ms. Roofline занижает в 6-10 раз.

Это ожидаемо — roofline даёт теоретический нижний предел. Реальность медленнее из-за:
- Маленькая модель (H=512) плохо утилизирует GPU — matmul'ы не насыщают SM'ы
- Overhead от запуска CUDA-ядер (~5-10 мкс каждое, сотни ядер за степ)
- Python-overhead: optimizer step, grad scaler, training loop
- Синхронизация DDP/FSDP

**Speedup**: 1.16x (calc) vs 1.92x (measured). Калькулятор недооценивает выигрыш, потому что не учитывает сокращение числа kernel launch'ей от fusion и разницу в optimizer step.